In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Parámetros de la red
v_p = 3.0
Sigma_p = 1.0
Sigma_u = 1.0
u = 2.0

def g(phi):
    return phi**2

def g_prime(phi):
    return 2 * phi



In [ ]:
# Valores posibles para phi y epsilon_p
phi_vals = np.linspace(0, 3.5, 20)
eps_p_vals = np.linspace(-3.0, 3.0, 20)
Phi, Eps_p = np.meshgrid(phi_vals, eps_p_vals)

# Asumimos epsilon_u estable
Eps_u_steady = (u - g(Phi)) / Sigma_u

# Copmutar derivadas para cada punto del espacio
dPhi = Eps_u_steady * g_prime(Phi) - Eps_p
dEps_p = Phi - v_p - Sigma_p * Eps_p

# Plotting the phase portrait
plt.figure(figsize=(8, 6))
plt.streamplot(Phi, Eps_p, dPhi, dEps_p, color=np.sqrt(dPhi**2 + dEps_p**2), cmap='viridis')

# Mark the optimal fixed point (approx phi=1.6 based on the exact posterior)
plt.plot(1.6, (1.6 - v_p)/Sigma_p, 'ro', markersize=8, label="Attractor (Optimal Belief)")

plt.xlabel(r'State Node ($\phi$)')
plt.ylabel(r'Prior Error Node ($\epsilon_p$)')
plt.title('Phase Portrait of Perceptual Inference')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Valores posibles para phi y epsilon_p
phi_vals = np.linspace(0, 3.5, 20)
eps_p_vals = np.linspace(-3.0, 3.0, 20)
eps_u_vals = np.linspace(-3.0, 3.0, 20)
Phi, Eps_p, Eps_u = np.meshgrid(phi_vals, eps_p_vals, eps_u_vals)


# Definimos la dinámica del sistema
def dynamics(t, state):
    phi, eps_p, eps_u = state
    return [
        eps_u * g_prime(phi) - eps_p,
        phi - v_p - Sigma_p * eps_p,
        u - g(phi) - Sigma_u * eps_u
    ]

# Initialized at the prior mean with zero starting error
sol = solve_ivp(dynamics, [0, 5], [3.0, 0.0, 0.0], dense_output=True)
t = np.linspace(0, 5, 300)
trajectory = sol.sol(t)

# 3. Calculate exact 3D Attractor coordinates
phi_star = 1.61 # Approximate optimal belief
eps_p_star = (phi_star - v_p) / Sigma_p
eps_u_star = (u - g(phi_star)) / Sigma_u

# 4. Plotting
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection='3d')

# Color the quiver field by vector magnitude
length = np.sqrt(dPhi**2 + dEps_p**2 + dEps_u**2)
q = ax.quiver(Phi, Eps_p, Eps_u, dPhi, dEps_p, dEps_u, 
              length=0.15, normalize=True, cmap='viridis', alpha=0.3)
q.set_array(length.flatten()) 

# Add Colorbar
cbar = fig.colorbar(q, ax=ax, shrink=0.5, pad=0.1)
cbar.set_label('Gradient Magnitude\n(Steepness of Energy Surface)')

# Overlay the integrated trajectory
ax.plot(trajectory[0], trajectory[1], trajectory[2], 
        color='black', linewidth=3, label="Inference Trajectory")

# Mark Initial State and Attractor
ax.plot([3.0], [0.0], [0.0], 'go', markersize=8, label="Start (Prior Belief)")
ax.plot([phi_star], [eps_p_star], [eps_u_star], 'r*', markersize=15, label="Attractor")

ax.set_xlabel(r'$\phi$')
ax.set_ylabel(r'$\epsilon_p$')
ax.set_zlabel(r'$\epsilon_u$')
plt.title('3D Phase Portrait & Neural Trajectory')
plt.legend()
plt.show()